<a href="https://colab.research.google.com/github/Sn0w616/Taller-Visualizacion-PowerBI/blob/main/Taller_Visualizacion_PowerBI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Taller práctico: ETL y visualización con Power BI
## Por: Camilo Andrés González López

**Objetivo:** extraer, perfilar, limpiar, integrar y cargar datos en un modelo dimensional listo para Power BI.

**Fuente:** archivo Excel `SuperStoreUS-2015.xlsx`, hojas `Orders`, `Returns` y `Users`.

**Repositorio en gitHub:** https://github.com/Sn0w616/Taller-Visualizacion-PowerBI

### Flujo de trabajo
1. Cargar el Excel original.
2. Examinar estructura y calidad de los datos.
3. Limpiar tipos, textos, fechas y valores faltantes.
4. Integrar devoluciones y responsables regionales.
5. Construir dimensiones y tabla de hechos.
6. Validar y exportar un Excel para Power BI.
7. Construir el tablero en Power BI para la visualización de datos.

## 0. Preparación del entorno y extracción
Se importan las librerías y se carga el dataset desde el equipo.

In [1]:
import os, zipfile, warnings
import pandas as pd
import numpy as np
from google.colab import files
warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 50)

uploaded = files.upload()
archivo = next((n for n in uploaded.keys() if n.lower().endswith((".xlsx", ".xls"))), None)
if archivo is None:
    raise FileNotFoundError("Debes cargar el dataset")

Saving SuperStoreUS-2015.xlsx to SuperStoreUS-2015.xlsx


## 1. Extracción y reconocimiento de las fuentes
Se leen las tres hojas sin alterar los datos de origen y se comprueba su tamaño.

In [2]:
# 1. EXTRACCIÓN
orders_raw = pd.read_excel(archivo, sheet_name="Orders")
returns_raw = pd.read_excel(archivo, sheet_name="Returns")
users_raw = pd.read_excel(archivo, sheet_name="Users")

print("EXTRACCIÓN")
for nombre, df in [("Orders", orders_raw), ("Returns", returns_raw), ("Users", users_raw)]:
    print(f"{nombre}: {df.shape[0]:,} filas y {df.shape[1]} columnas")

display(orders_raw.head())
display(returns_raw.head())
display(users_raw)

EXTRACCIÓN
Orders: 1,952 filas y 25 columnas
Returns: 1,634 filas y 2 columnas
Users: 4 filas y 2 columnas


,Row ID,Order Priority,Discount,Unit Price,Shipping Cost,Customer ID,Customer Name,Ship Mode,Customer Segment,Product Category,Product Sub-Category,Product Container,Product Name,Product Base Margin,Country,Region,State or Province,City,Postal Code,Order Date,Ship Date,Profit,Quantity ordered new,Sales,Order ID
0,20847,High,0.01,2.84,0.93,3,Bonnie Potter,Express Air,Corporate,Office Supplies,Pens & Art Supplies,Wrap Bag,SANFORD Liquid Accent™ Tank-Style Highlighters,0.54,United States,West,Washington,Anacortes,98221,2015-01-07,2015-01-08,4.5600,4,13.01,88522
1,20228,Not Specified,0.02,500.98,26.00,5,Ronnie Proctor,Delivery Truck,Home Office,Furniture,Chairs & Chairmats,Jumbo Drum,Global Troy™ Executive Leather Low-Back Tilter,0.60,United States,West,California,San Gabriel,91776,2015-06-13,2015-06-15,4390.3665,12,6362.85,90193
2,21776,Critical,0.06,9.48,7.29,11,Marcus Dunlap,Regular Air,Home Office,Furniture,Office Furnishings,Small Pack,"DAX Two-Tone Rosewood/Black Document Frame, De...",0.45,United States,East,New Jersey,Roselle,7203,2015-02-15,2015-02-17,-53.8096,22,211.15,90192
3,24844,Medium,0.09,78.69,19.99,14,Gwendolyn F Tyson,Regular Air,Small Business,Furniture,Office Furnishings,Small Box,Howard Miller 12-3/4 Diameter Accuwave DS ™ Wa...,0.43,United States,Central,Minnesota,Prior Lake,55372,2015-05-12,2015-05-14,803.4705,16,1164.45,86838
4,24846,Medium,0.08,3.28,2.31,14,Gwendolyn F Tyson,Regular Air,Small Business,Office Supplies,Pens & Art Supplies,Wrap Bag,Newell 321,0.56,United States,Central,Minnesota,Prior Lake,55372,2015-05-12,2015-05-13,-24.0300,7,22.23,86838


,Order ID,Status
0,65,Returned
1,612,Returned
2,614,Returned
3,678,Returned
4,710,Returned


,Region,Manager
0,Central,Chris
1,East,Erin
2,South,Sam
3,West,William


## 2. Perfilamiento de calidad
Se identifican tipos, valores nulos, cardinalidad y duplicados exactos. El perfil queda exportado en la hoja `DataQuality`.

In [3]:
# 2. PERFILAMIENTO DE CALIDAD
def perfil_calidad(df, nombre):
    return pd.DataFrame({
        "tabla": nombre,
        "tipo_dato": df.dtypes.astype(str),
        "nulos": df.isna().sum(),
        "porcentaje_nulos": (df.isna().mean()*100).round(2),
        "valores_unicos": df.nunique(dropna=True),
        "duplicados_exactos": [df.duplicated().sum()]*len(df.columns)
    }).rename_axis("columna").reset_index()

perfil = pd.concat([
    perfil_calidad(orders_raw, "Orders"),
    perfil_calidad(returns_raw, "Returns"),
    perfil_calidad(users_raw, "Users")
], ignore_index=True)
display(perfil)

,columna,tabla,tipo_dato,nulos,porcentaje_nulos,valores_unicos,duplicados_exactos
0,Row ID,Orders,int64,0,0.00,1951,0
1,Order Priority,Orders,object,0,0.00,6,0
2,Discount,Orders,float64,0,0.00,13,0
3,Unit Price,Orders,float64,0,0.00,597,0
4,Shipping Cost,Orders,float64,0,0.00,497,0
5,Customer ID,Orders,int64,0,0.00,1130,0
6,Customer Name,Orders,object,0,0.00,1130,0
7,Ship Mode,Orders,object,0,0.00,3,0
8,Customer Segment,Orders,object,0,0.00,4,0
9,Product Category,Orders,object,0,0.00,3,0


## 3. Transformación y limpieza
Se normalizan fechas, tipos numéricos y textos. Los 16 valores nulos de `Product Base Margin` se imputan primero con la mediana de la subcategoría y luego con la mediana global. Se conserva la trazabilidad creando `OrderLineID`, ya que `Row ID` aparece repetido en el origen.

In [4]:
# 3. TRANSFORMACIÓN
orders = orders_raw.copy()
returns = returns_raw.copy()
users = users_raw.copy()

# Normalizar encabezados: nombres uniformes, sin espacios ni signos
def normalizar_columnas(df):
    df = df.copy()
    df.columns = [str(c).strip() for c in df.columns]
    return df

orders, returns, users = map(normalizar_columnas, [orders, returns, users])

# Conversión de fechas
for col in ["Order Date", "Ship Date"]:
    orders[col] = pd.to_datetime(orders[col], errors="coerce")

# Conversión numérica explícita
columnas_numericas = ["Discount", "Unit Price", "Shipping Cost", "Product Base Margin",
                      "Profit", "Quantity ordered new", "Sales", "Customer ID",
                      "Postal Code", "Row ID", "Order ID"]
for col in columnas_numericas:
    orders[col] = pd.to_numeric(orders[col], errors="coerce")

returns["Order ID"] = pd.to_numeric(returns["Order ID"], errors="coerce")
users["Region"] = users["Region"].astype("string").str.strip().str.title()
users["Manager"] = users["Manager"].astype("string").str.strip()

# Eliminar duplicados exactos y asegurar identificador único de línea
n_dup = int(orders.duplicated().sum())
orders = orders.drop_duplicates().copy()
orders = orders.dropna(subset=["Row ID", "Order ID", "Order Date", "Customer ID"]).copy()
orders["Row ID"] = orders["Row ID"].astype("int64")
# Row ID se repite en el archivo fuente; se crea una llave sustituta única por línea.
orders = orders.reset_index(drop=True)
orders["LineKey"] = np.arange(1, len(orders) + 1)
orders["Order ID"] = orders["Order ID"].astype("int64")
orders["Customer ID"] = orders["Customer ID"].astype("int64")

# Limpieza de textos
text_cols = ["Order Priority", "Ship Mode", "Customer Name", "Customer Segment",
             "Product Category", "Product Sub-Category", "Product Container",
             "Product Name", "Country", "Region", "State or Province", "City"]
for col in text_cols:
    orders[col] = orders[col].astype("string").str.strip()
    orders[col] = orders[col].replace({"": pd.NA, "nan": pd.NA, "None": pd.NA})

# Valores faltantes: margen de producto se conserva como nulo y se documenta;
# no se inventan valores para variables comerciales.
orders["Product Base Margin"] = orders["Product Base Margin"].fillna(
    orders.groupby("Product Sub-Category")["Product Base Margin"].transform("median")
)
orders["Product Base Margin"] = orders["Product Base Margin"].fillna(
    orders["Product Base Margin"].median()
)

# Variables derivadas
orders["Order Year"] = orders["Order Date"].dt.year
orders["Order Month Number"] = orders["Order Date"].dt.month
orders["Order Month"] = orders["Order Date"].dt.strftime("%B")
orders["Order Year Month"] = orders["Order Date"].dt.strftime("%Y-%m")
orders["Shipping Days"] = (orders["Ship Date"] - orders["Order Date"]).dt.days
orders["Return Status"] = "Not Returned"

## 4. Integración de fuentes
Las devoluciones se agregan por `Order ID` antes de unirlas, evitando multiplicar líneas de venta. Los responsables se integran por región. Se reportan las coincidencias y los identificadores de devoluciones que no aparecen en Orders.

In [5]:
# 4. INTEGRACIÓN: devoluciones y responsables regionales
# Returns se resume por Order ID para evitar multiplicar líneas de venta.
returns_valid = returns.dropna(subset=["Order ID"]).copy()
returns_valid["Order ID"] = returns_valid["Order ID"].astype("int64")
returns_by_order = (returns_valid.groupby("Order ID")
                    .agg(Return_Status=("Status", lambda x: ", ".join(sorted(set(x.dropna().astype(str))))))
                    .reset_index())
orders = orders.merge(returns_by_order, on="Order ID", how="left")
orders["Return Status"] = orders["Return_Status"].fillna("Not Returned")
orders.drop(columns=["Return_Status"], inplace=True)

orders = orders.merge(users.rename(columns={"Manager":"Regional Manager"}),
                      on="Region", how="left")
orders["Regional Manager"] = orders["Regional Manager"].fillna("Unassigned")

# Identificar calidad de integración de devoluciones
ids_orders = set(orders["Order ID"].unique())
ids_returns = set(returns_valid["Order ID"].unique())
coincidentes = len(ids_orders & ids_returns)
no_coincidentes = len(ids_returns - ids_orders)
print(f"Órdenes con devolución identificadas: {coincidentes}")
print(f"IDs de Returns sin coincidencia en Orders: {no_coincidentes}")

Órdenes con devolución identificadas: 11
IDs de Returns sin coincidencia en Orders: 1623


## 5. Construcción del modelo dimensional
Se construyen dimensiones de fecha, cliente, producto, geografía, modo de envío y prioridad, junto con `FactSales`. La tabla de hechos conserva las métricas de venta, ganancia, cantidad, descuento y costo de envío.

In [6]:
# 5. MODELO DIMENSIONAL (modelo estrella)
# Dimensión fecha: calendario continuo entre fechas mínimas y máximas
fechas = pd.date_range(orders["Order Date"].min(), orders["Order Date"].max(), freq="D")
dim_fecha = pd.DataFrame({"Date": fechas})
dim_fecha["DateKey"] = dim_fecha["Date"].dt.strftime("%Y%m%d").astype("int64")
dim_fecha["Year"] = dim_fecha["Date"].dt.year
dim_fecha["Quarter"] = "Q" + dim_fecha["Date"].dt.quarter.astype(str)
dim_fecha["MonthNumber"] = dim_fecha["Date"].dt.month
dim_fecha["MonthName"] = dim_fecha["Date"].dt.strftime("%B")
dim_fecha["YearMonth"] = dim_fecha["Date"].dt.strftime("%Y-%m")
dim_fecha["Day"] = dim_fecha["Date"].dt.day
dim_fecha["Weekday"] = dim_fecha["Date"].dt.strftime("%A")

# Dimensión cliente
dim_cliente = orders[["Customer ID", "Customer Name", "Customer Segment"]].drop_duplicates("Customer ID").copy()
dim_cliente = dim_cliente.rename(columns={"Customer ID":"CustomerKey", "Customer Name":"CustomerName",
                                          "Customer Segment":"CustomerSegment"})

# Dimensión producto
prod_cols = ["Product Category", "Product Sub-Category", "Product Name", "Product Container",
             "Product Base Margin", "Unit Price"]
dim_producto = orders[prod_cols].drop_duplicates("Product Name").copy()
dim_producto = dim_producto.rename(columns={
    "Product Category":"ProductCategory", "Product Sub-Category":"ProductSubCategory",
    "Product Name":"ProductName", "Product Container":"ProductContainer",
    "Product Base Margin":"ProductBaseMargin", "Unit Price":"UnitPrice"
})
dim_producto.insert(0, "ProductKey", range(1, len(dim_producto)+1))

# Dimensión geográfica
geo_cols = ["Country", "Region", "State or Province", "City", "Postal Code", "Regional Manager"]
dim_geografia = orders[geo_cols].drop_duplicates().copy().reset_index(drop=True)
dim_geografia.insert(0, "GeographyKey", range(1, len(dim_geografia)+1))
dim_geografia = dim_geografia.rename(columns={"State or Province":"StateProvince",
                                              "Postal Code":"PostalCode",
                                              "Regional Manager":"RegionalManager"})

# Dimensiones operativas
dim_envio = pd.DataFrame({"ShipMode": sorted(orders["Ship Mode"].dropna().unique())})
dim_envio.insert(0, "ShipModeKey", range(1, len(dim_envio)+1))
dim_prioridad = pd.DataFrame({"OrderPriority": sorted(orders["Order Priority"].dropna().unique())})
dim_prioridad.insert(0, "PriorityKey", range(1, len(dim_prioridad)+1))

# Construir claves foráneas en la tabla de hechos
fact = orders.copy()
fact["DateKey"] = fact["Order Date"].dt.strftime("%Y%m%d").astype("int64")
fact = fact.merge(dim_producto[["ProductKey","ProductName"]].rename(columns={"ProductName":"Product Name"}),
                  on="Product Name", how="left")
geo_lookup = dim_geografia.rename(columns={"StateProvince":"State or Province", "PostalCode":"Postal Code", "RegionalManager":"Regional Manager"})
fact = fact.merge(geo_lookup[geo_cols+["GeographyKey"]], on=geo_cols, how="left")
fact = fact.merge(dim_envio, left_on="Ship Mode", right_on="ShipMode", how="left")
fact = fact.merge(dim_prioridad, left_on="Order Priority", right_on="OrderPriority", how="left")

# Fact con métricas al nivel de línea de pedido (Row ID)
fact_ventas = fact[[
    "LineKey","Row ID","Order ID","Customer ID","ProductKey","DateKey","GeographyKey",
    "ShipModeKey","PriorityKey","Discount","Unit Price","Shipping Cost",
    "Product Base Margin","Profit","Quantity ordered new","Sales",
    "Shipping Days","Return Status"
]].copy()
fact_ventas = fact_ventas.rename(columns={
    "LineKey":"OrderLineID","Row ID":"SourceRowID","Order ID":"OrderID","Customer ID":"CustomerKey",
    "Discount":"DiscountRate","Unit Price":"UnitPrice","Shipping Cost":"ShippingCost",
    "Product Base Margin":"ProductBaseMargin","Profit":"Profit",
    "Quantity ordered new":"Quantity","Sales":"Sales"
})

# Dimensiones con nombres estandarizados
dim_fecha = dim_fecha.rename(columns={"Date":"FullDate"})
dim_fecha["FullDate"] = pd.to_datetime(dim_fecha["FullDate"])
dim_cliente = dim_cliente.sort_values("CustomerKey").reset_index(drop=True)

## 6. Validaciones del ETL
Se verifica la unicidad de la llave sustituta, la integridad de las llaves foráneas y la disponibilidad de las métricas.

In [7]:
# 6. VALIDACIONES
assert fact_ventas["OrderLineID"].is_unique, "OrderLineID debe ser único"
assert fact_ventas[["CustomerKey","ProductKey","DateKey","GeographyKey","ShipModeKey","PriorityKey"]].notna().all().all(), "Hay llaves foráneas nulas"
assert (fact_ventas["Quantity"] >= 0).all(), "Se encontraron cantidades negativas"
assert fact_ventas["Sales"].notna().all(), "Hay ventas nulas"

resumen_validacion = pd.DataFrame({
    "Indicador": ["Filas originales Orders", "Filas después de limpieza", "Duplicados eliminados",
                  "Líneas de venta cargadas", "Órdenes únicas", "Clientes", "Productos",
                  "Registros de devolución coincidentes", "IDs de Returns sin coincidencia",
                  "Ventas totales", "Ganancia total"],
    "Valor": [len(orders_raw), len(orders), n_dup, len(fact_ventas),
              fact_ventas["OrderID"].nunique(), len(dim_cliente), len(dim_producto),
              coincidentes, no_coincidentes, fact_ventas["Sales"].sum(), fact_ventas["Profit"].sum()]
})
display(resumen_validacion)

,Indicador,Valor
0,Filas originales Orders,1.952000e+03
1,Filas después de limpieza,1.952000e+03
2,Duplicados eliminados,0.000000e+00
3,Líneas de venta cargadas,1.952000e+03
4,Órdenes únicas,1.365000e+03
5,Clientes,1.130000e+03
6,Productos,9.130000e+02
7,Registros de devolución coincidentes,1.100000e+01
8,IDs de Returns sin coincidencia,1.623000e+03
9,Ventas totales,1.924338e+06


## 7. Carga y exportación
Se genera `SuperStoreUS_2015_ETL_PowerBI.xlsx`, con una hoja por tabla del modelo y hojas de auditoría.

In [8]:
# 7. CARGA: exportar modelo para Power BI
salida = "SuperStoreUS_2015_ETL_PowerBI.xlsx"
with pd.ExcelWriter(salida, engine="openpyxl") as writer:
    fact_ventas.to_excel(writer, sheet_name="FactSales", index=False)
    dim_cliente.to_excel(writer, sheet_name="DimCustomer", index=False)
    dim_producto.to_excel(writer, sheet_name="DimProduct", index=False)
    dim_fecha.to_excel(writer, sheet_name="DimDate", index=False)
    dim_geografia.to_excel(writer, sheet_name="DimGeography", index=False)
    dim_envio.to_excel(writer, sheet_name="DimShipMode", index=False)
    dim_prioridad.to_excel(writer, sheet_name="DimPriority", index=False)
    perfil.to_excel(writer, sheet_name="DataQuality", index=False)
    resumen_validacion.to_excel(writer, sheet_name="ETL_Summary", index=False)
    returns_raw.to_excel(writer, sheet_name="Source_Returns", index=False)

# CSV adicional de hechos por si se requiere
fact_ventas.to_csv("FactSales_SuperStore.csv", index=False, encoding="utf-8-sig")
print(f"Archivo generado: {salida}")
files.download(salida)

Archivo generado: SuperStoreUS_2015_ETL_PowerBI.xlsx


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 8. Power BI Desktop: Construcción del tablero

### 8.1 Importar el resultado del ETL
1. Abre Power BI Desktop.
2. Selecciona **Inicio → Obtener datos → Libro de Excel**.
3. Selecciona `SuperStoreUS_2015_ETL_PowerBI.xlsx`.
4. En el Navegador marca `FactSales`, `DimCustomer`, `DimProduct`, `DimDate`, `DimGeography`, `DimShipMode` y `DimPriority`.
5. Pulsa **Cargar**.

### 8.2 Configurar relaciones (Vista Modelo)
Crea relaciones de **uno a varios (1:*)**, con filtro en una sola dirección desde dimensión hacia hechos:

| Dimensión (lado 1) | Campo | Hechos (lado *) | Campo |
|---|---|---|---|
| DimCustomer | CustomerKey | FactSales | CustomerKey |
| DimProduct | ProductKey | FactSales | ProductKey |
| DimDate | DateKey | FactSales | DateKey |
| DimGeography | GeographyKey | FactSales | GeographyKey |
| DimShipMode | ShipModeKey | FactSales | ShipModeKey |
| DimPriority | PriorityKey | FactSales | PriorityKey |

En **Herramientas de tabla**, se marca `DimDate` como tabla de fechas usando `FullDate`.

### 8.3 Ordenar los campos temporales
- Selecciona `DimDate[MonthName]` → **Ordenar por columna** → `MonthNumber`.
- Para los gráficos cronológicos utiliza `DimDate[YearMonth]` y ordénalo ascendentemente. Es un campo `AAAA-MM`, por lo que conserva el orden temporal.

### 8.4 Crear medidas DAX
Selecciona la tabla `FactSales` → **Nueva medida**. Crea cada medida por separado:

```dax
Ventas Totales = SUM(FactSales[Sales])

Ganancia Total = SUM(FactSales[Profit])

Unidades Vendidas = SUM(FactSales[Quantity])

Órdenes = DISTINCTCOUNT(FactSales[OrderID])

Líneas de Pedido = COUNTROWS(FactSales)

Ticket Promedio = DIVIDE([Ventas Totales], [Órdenes], 0)

Margen de Ganancia % = DIVIDE([Ganancia Total], [Ventas Totales], 0)

Descuento Promedio = AVERAGE(FactSales[DiscountRate])

Costo de Envío = SUM(FactSales[ShippingCost])

Días Promedio de Envío = AVERAGE(FactSales[Shipping Days])

Líneas Devueltas =
CALCULATE(
    COUNTROWS(FactSales),
    FactSales[Return Status] <> "Not Returned"
)

% Líneas Devueltas =
DIVIDE([Líneas Devueltas], [Líneas de Pedido], 0)

Ventas Acumuladas YTD =
TOTALYTD([Ventas Totales], DimDate[FullDate])
```

Da formato de moneda a ventas, ganancias, ticket y costo de envío; formato porcentaje a margen, descuento y devoluciones; y formato decimal a promedios.

### 8.5 Diseño del tablero de revisión
Construye una página principal llamada **Resumen ejecutivo** con estos objetos:

| Zona | Visual | Campos |
|---|---|---|
| Superior | Tarjetas KPI (5) | Ventas Totales, Ganancia Total, Órdenes, Unidades Vendidas, Margen de Ganancia % |
| Centro superior | Línea temporal | Eje: DimDate[YearMonth]; valores: Ventas Totales y Ganancia Total |
| Centro izquierdo | Barras agrupadas | Eje: DimProduct[ProductCategory]; valores: Ventas Totales y Ganancia Total |
| Centro derecho | Barras horizontales | Eje: DimProduct[ProductSubCategory]; valores: Ventas Totales |
| Inferior izquierdo | Dona | Leyenda: DimCustomer[CustomerSegment]; valores: Ventas Totales |
| Inferior centro | Barras | Eje: DimGeography[Region]; valores: Ventas Totales; leyenda: DimGeography[RegionalManager] |
| Inferior derecho | Dona o barras | Leyenda: DimShipMode[ShipMode]; valores: Órdenes |

Agrega segmentadores para `DimDate[Year]`, `DimGeography[Region]`, `DimCustomer[CustomerSegment]`, `DimProduct[ProductCategory]` y `DimShipMode[ShipMode]`.

### 8.6 Página de producto y rentabilidad
Crea una segunda página llamada **Productos y rentabilidad**:
- Barras: ventas por categoría y subcategoría.
- Tabla o matriz: ProductName, Ventas Totales, Ganancia Total, Unidades Vendidas y Margen de Ganancia %.
- Dispersión: eje X = Ventas Totales, eje Y = Ganancia Total, detalles = ProductName; tamaño = Unidades Vendidas.
- Tarjetas: Descuento Promedio, Costo de Envío y Margen de Ganancia %.

### 8.7 Página de logística y devoluciones
Crea una tercera página llamada **Logística y devoluciones**:
- Barras: Órdenes por ShipMode.
- Columnas: Días Promedio de Envío por ShipMode.
- Tarjetas: Líneas Devueltas y % Líneas Devueltas.
- Tabla: OrderID, Return Status, ShipMode, Shipping Days, Sales y Profit.
- Segmentadores: región, prioridad y modo de envío.

**Nota de interpretación:** la hoja `Returns` contiene 1.634 identificadores, pero solo 11 coinciden con los `Order ID` presentes en `Orders`. Por tanto, el indicador de devoluciones refleja únicamente coincidencias verificables entre ambas hojas. No interpretes el resto como devoluciones confirmadas de las órdenes de esta tabla; conviene revisar el origen de esos identificadores.

### 8.8 Página de calidad y auditoría (opcional)
Si importas `DataQuality` y `ETL_Summary`, crea una página con:
- Tabla de nulos y porcentaje de nulos por columna.
- Tarjetas con filas originales, filas cargadas, duplicados eliminados y órdenes con devolución coincidentes.
- Tabla de conciliación con ventas y ganancias totales.


### 8.10 Guardar y entregar
Guarda el informe como `Tablero_SuperStoreUS_2015.pbix`. Conserva también el Excel ETL como fuente de datos. Al actualizar el informe, utiliza **Inicio → Actualizar** después de reemplazar el archivo fuente por una versión con la misma estructura.